<a href="https://colab.research.google.com/github/muneer-ahmad10/Chatbot/blob/main/RAG_ChatBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install streamlit pypdf sentence-transformers faiss-cpu transformers accelerate

In [ ]:
from pypdf import PdfReader

def extract_text(pdf_file):

    reader = PdfReader(pdf_file)

    text = ""

    for page in reader.pages:
        text += page.extract_text()

    return text

In [ ]:
pdf_filename = "/content/Polymorphism.pdf"

pdf_text = extract_text(pdf_filename)

In [ ]:
print(pdf_text[:600])

In [ ]:
def create_chunks(text, chunk_size=500):

    chunks = []

    for i in range(0, len(text), chunk_size):

        chunk = text[i:i+chunk_size]

        chunks.append(chunk)

    return chunks

In [ ]:
chunks = create_chunks(pdf_text)
print("Total Chunks:", len(chunks))

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

chunk_embeddings = embedding_model.encode(chunks)

In [ ]:
print(chunk_embeddings.shape)

In [ ]:
import faiss
import numpy as np

dimension = chunk_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    np.array(chunk_embeddings).astype("float32")
)

In [ ]:
print(index.ntotal)

In [ ]:
len(chunks)

In [ ]:
def retrieve_context(question):

    question_embedding = embedding_model.encode(
        [question]
    )

    distances, indices = index.search(
        np.array(question_embedding).astype("float32"),
        k=3
    )

    retrieved_chunks = []

    for idx in indices[0]:

        if idx != -1:

            retrieved_chunks.append(
                chunks[idx]
            )

    return "\n".join(retrieved_chunks)

In [ ]:
context = retrieve_context(
    "What is Polymorphism?"
)

print(context)

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

llm = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)

In [ ]:
def answer_question(question):

    context = retrieve_context(question)

    prompt = f"""
    Answer the question using only
    the provided context.

    Context:
    {context}

    Question:
    {question}

    Answer:
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    outputs = llm.generate(
        **inputs,
        max_new_tokens=100
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [ ]:
question = "What is Polymorphism?"

answer = answer_question(
    question
)

print(answer)

# **Creating APP.py**

In [1]:
%%writefile app.py

import streamlit as st
import numpy as np
import faiss

from pypdf import PdfReader

from sentence_transformers import SentenceTransformer

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)



st.set_page_config(page_title="PDF Chatbot")

st.title("📚 PDF RAG Chatbot")

st.write("Upload a PDF and ask questions.")



@st.cache_resource
def load_models():

    embedding_model = SentenceTransformer(
        "all-MiniLM-L6-v2"
    )

    tokenizer = AutoTokenizer.from_pretrained(
        "google/flan-t5-base"
    )

    llm = AutoModelForSeq2SeqLM.from_pretrained(
        "google/flan-t5-base"
    )

    return embedding_model, tokenizer, llm




embedding_model, tokenizer, llm = load_models()



uploaded_file = st.file_uploader(
    "Upload PDF",
    type=["pdf"]
)



def extract_text(pdf_file):

    reader = PdfReader(pdf_file)

    text = ""

    for page in reader.pages:

        page_text = page.extract_text()

        if page_text:

            text += page_text

    return text


def create_chunks(text, chunk_size=500):

    chunks = []

    for i in range(0, len(text), chunk_size):

        chunks.append(
            text[i:i+chunk_size]
        )

    return chunks



if uploaded_file:
  pdf_text = extract_text(
        uploaded_file
    )

  chunks = create_chunks(
        pdf_text
    )

  chunk_embeddings = embedding_model.encode(
        chunks
    )




dimension = chunk_embeddings.shape[1]

index = faiss.IndexFlatL2(
    dimension
)

index.add(
    np.array(chunk_embeddings)
    .astype("float32")
)



st.success(
    f"Indexed {len(chunks)} chunks."
)


question = st.text_input(
    "Ask a question"
)


def retrieve_context(question):

    q_embedding = embedding_model.encode(
        [question]
    )

    distances, indices = index.search(
        np.array(q_embedding)
        .astype("float32"),
        k=3
    )

    retrieved = []

    for idx in indices[0]:

        if idx != -1:

            retrieved.append(
                chunks[idx]
            )

    return "\n".join(retrieved)





def answer_question(question):

    context = retrieve_context(
        question
    )

    prompt = f"""
    Answer the question using only the context.

    Context:
    {context}

    Question:
    {question}

    Answer:
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    outputs = llm.generate(
        **inputs,
        max_new_tokens=100
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer








if question:

    with st.spinner(
        "Thinking..."
    ):

        answer = answer_question(
            question
        )

    st.subheader(
        "Answer"
    )

    st.write(answer)








Writing app.py
